In [1]:
# PREAMBLE CELL — run this first after a Kernel Restart
import os, sys, platform
print("Py exe:", sys.executable)
print("Arch:", platform.machine())  # should be arm64 and point to your .venv

# Make sure XLA is OFF (Metal doesn't support XLA/JIT)
os.environ.pop("TF_XLA_FLAGS", None)
os.environ.pop("XLA_FLAGS", None)
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"

# Optional: quieter logs
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
tf.config.optimizer.set_jit(False)  # belt & suspenders

print("GPUs:", tf.config.list_physical_devices("GPU"))

m = tf.keras.Sequential([
    tf.keras.layers.Input((101,60)),
    tf.keras.layers.Conv1D(64,3,padding="same",activation="relu"),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32")
])
m.compile(optimizer="adam", loss="binary_crossentropy", jit_compile=False)
import numpy as np
x = np.random.randn(32,101,60).astype("float32")
y = np.random.randint(0,2,(32,1)).astype("float32")
m.fit(x,y,epochs=1,verbose=1)

Py exe: /Users/adrianzapaterreig/Documents/Personal/TFM/rickd-analysis/.venv/bin/python
Arch: arm64
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


I0000 00:00:1755470744.409590  568080 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1755470744.409609  568080 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 422ms/step - loss: 0.7040
